In [2]:
from semantic_odds import generate_semantic_report

generate_semantic_report(
    image_url="https://neroexample.oss-cn-hangzhou.aliyuncs.com/fei_wuda.png",
    file_name="yj_95_odds",
    skip_ocr=True,
    save_md=True,
    docs_dir="../docs/odds",
)

⚙️ 已跳过图片识别阶段
🧩 开始语义化处理：../docs/odds/yj_95_odds.md
✅ 语义化完成，已输出至：../docs/odds/yj_95_odds_lang.md

🎯 全部流程完成 ✅
输出文件：../docs/odds/yj_95_odds_lang.md


'../docs/odds/yj_95_odds_lang.md'

In [9]:
! pip install -U unstructured markdown

In [23]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.vectorstores import FAISS
import os

load_dotenv()

def vectorize_markdown(file_name: str, mode:str = "elements") -> str:
    """向量化 markdown 文件"""
    markdown_path = "../docs/odds/" + file_name + ".md"
    # 使用 markdown loader 加载 markdown
    # elements 模式非常适合 RAG
    # 可以把长 Markdown 拆分成更小的段落向量化，提高检索精度。
    loader = UnstructuredMarkdownLoader(
        markdown_path,
        mode=mode,
    )
    data = loader.load()
    # 使用向量模型
    embeddings = DashScopeEmbeddings(
        model="text-embedding-v3",
        dashscope_api_key=os.getenv("QWEN_API_KEY"),
    )

    db = FAISS.from_documents(data, embeddings)
    db.save_local("../data/vectordb/faiss_" + file_name)
    print("✅ 已构建并保存 FAISS 向量数据库")

vectorize_markdown("yj_95_odds_lang")
vectorize_markdown("yj_95_odds", mode="single")


✅ 已构建并保存 FAISS 向量数据库
✅ 已构建并保存 FAISS 向量数据库


In [52]:
embeddings = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=os.getenv("QWEN_API_KEY"),
)

db = FAISS.load_local(
    "../data/vectordb/faiss_yj_95_odds_lang",
    embeddings,
    allow_dangerous_deserialization=True,
)


def get_answer(query: str):
    docs = db.similarity_search(query, k=2)
    print(query)
    print("""---- 关联内容 ----""")
    for i, doc in enumerate(docs):
        page_content = doc.page_content
        text = page_content[: len(page_content)]
        print(text.replace("。", "\n"))

# get_answer("1.5球中水对应的HAD是多少？")
# get_answer("1球高水对应的HAD是多少？")
get_answer("亚盘主队让1球")
get_answer("亚盘主队让1.25球")
get_answer("亚盘主队让0.5球")

亚盘主队让1球
---- 关联内容 ----
5区的主胜赔率（H）从1.5上升到1.63，平局赔率（D）在4.0到4.2之间，客胜赔率（A）从5.25降至6.5
这表明主队仍占优势，但客队获胜机会略有增加
返还率约为94.5%，保持稳定
亚洲盘口为主队让1球，水位从低到高：低水位时主队投注较安全，高水位则可能反映主队优势减弱，需注意风险

1区的主胜赔率（H）从2.45到2.55，平局赔率（D）保持3.4，客胜赔率（A）从2.7到2.8
这表示比赛势均力敌，主客队获胜机会相近
返还率约为94.5%
亚洲盘口为平手盘（-0），水位低，说明投注主队回报较低，但风险相对可控

亚盘主队让1.25球
---- 关联内容 ----
5区的主胜赔率（H）从1.5上升到1.63，平局赔率（D）在4.0到4.2之间，客胜赔率（A）从5.25降至6.5
这表明主队仍占优势，但客队获胜机会略有增加
返还率约为94.5%，保持稳定
亚洲盘口为主队让1球，水位从低到高：低水位时主队投注较安全，高水位则可能反映主队优势减弱，需注意风险

2区的主胜赔率（H）范围从2.1到2.4，平局赔率（D）稳定在3.4，客胜赔率（A）从2.9到3.5
主队优势不明显，客胜概率上升
返还率在94%至95%之间
亚洲盘口为主队让0.25球，水位从低到高：低水位时主队稍占上风，高水位则强调比赛可能以平局或客胜收场

亚盘主队让0.5球
---- 关联内容 ----
5区的主胜赔率（H）从1.5上升到1.63，平局赔率（D）在4.0到4.2之间，客胜赔率（A）从5.25降至6.5
这表明主队仍占优势，但客队获胜机会略有增加
返还率约为94.5%，保持稳定
亚洲盘口为主队让1球，水位从低到高：低水位时主队投注较安全，高水位则可能反映主队优势减弱，需注意风险

4区的主胜赔率（H）范围从1.65到1.83，平局赔率（D）从3.75到4.0，客胜赔率（A）从4.2到5.0
主队优势进一步减弱，比赛结果更趋于平衡
返还率在94.5%左右波动
亚洲盘口为主队让0.75球，水位从低到高：低水位表示主队赔率较稳健，高水位则提示平局或客胜可能性增加

